# 01. Оптимальное программное управление


In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base = Path.cwd().resolve()
if (base / 'src').exists():
    ROOT = base
elif (base / 'research' / 'src').exists():
    ROOT = base / 'research'
elif (base.parent / 'src').exists():
    ROOT = base.parent
else:
    raise RuntimeError('Cannot locate research/src directory from current working directory')

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

plt.style.use('seaborn-v0_8-whitegrid')

In [4]:
from dynamics import CostConfig, build_trajectory_bundle, bolza_cost, load_training_samples, anneal


DATA_DIR = ROOT / 'src' / 'data'
samples = load_training_samples(DATA_DIR)
bundle = build_trajectory_bundle(samples)

print('Samples:', samples.shape)
print('Trajectories with scores:', len(bundle))
print('Steps range:', int(samples['step'].min()), '..', int(samples['step'].max()))
print(samples.head(3))


Samples: (16170, 13)
Trajectories with scores: 1078
Steps range: 1 .. 15
   trajectory_id  step        x1        x2        x3        x4        x5  \
0              1     1 -0.200000 -0.200000 -0.200000  0.000000  0.000000   
1              1     2 -0.510875 -0.275485 -0.125600 -1.665400 -0.404384   
2              1     3 -1.410614 -0.213993  0.095575 -3.154632  0.733806   

         x6        u1        u2        u3         u4      score  
0  0.000000  0.187504 -0.354690 -0.261799  11.806562  10.717483  
1  0.398569 -0.202423 -0.385951 -0.153431  11.954005  10.717483  
2  0.786299 -0.224996 -0.030610 -0.176636  10.474055  10.717483  


In [5]:
score_by_traj = samples[['trajectory_id', 'score']].drop_duplicates().sort_values('score')
best_tid = int(score_by_traj.iloc[0]['trajectory_id'])
worst_tid = int(score_by_traj.iloc[-1]['trajectory_id'])

print('Best observed trajectory id:', best_tid)
print('Best observed score:', float(score_by_traj.iloc[0]['score']))
print('Worst observed score:', float(score_by_traj.iloc[-1]['score']))
print('Top-10 observed trajectories:')
print(score_by_traj.head(10).to_string(index=False))


Best observed trajectory id: 1040
Best observed score: 5.628439447449216
Worst observed score: 180.69447547526855
Top-10 observed trajectories:
 trajectory_id    score
          1040 5.628439
           162 5.643882
           687 5.646878
          1000 5.657905
           842 5.658480
          1047 5.659009
           932 5.661025
           957 5.661259
           890 5.663216
           662 5.664261


In [6]:
cfg = CostConfig()
best_obs = bundle[best_tid]
X_obs = best_obs['X']
U_obs = best_obs['U']

calc_score = bolza_cost(X_obs[0], U_obs, cfg)
print('Best observed score from DB:', best_obs['score'])
print('Recomputed score (paper functional):', calc_score)
print('Absolute difference:', abs(best_obs['score'] - calc_score))


Best observed score from DB: 5.628439447449216
Recomputed score (paper functional): 5.628439447449216
Absolute difference: 0.0


In [ ]:
# Simulated annealing for one selected initial state.
# We optimize controls directly and keep solver-compatible bounds.
def random_controls(n_steps: int, seed: int = 39) -> np.ndarray:
    rng = np.random.default_rng(seed)
    low = np.array([-np.pi/12, -np.pi, -np.pi/12, 0.0])
    high = np.array([ np.pi/12,  np.pi,  np.pi/12, 12.0])
    return low + (high - low) * rng.random((n_steps, 4))

initial_state = X_obs[0].copy()
start_u = random_controls(15, seed=39)

best_u_opt, best_j_opt, trace = anneal(initial_state, start_u, cfg, n_iter=300000, seed=39)
print('Observed best score for selected initial state:', best_obs['score'])
print('Optimized score from annealing:', best_j_opt)
print('Improvement:', float(best_obs['score'] - best_j_opt))

In [ ]:
from dynamics import rollout

X_opt = rollout(initial_state, best_u_opt, cfg.dt)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].plot(X_opt[:, 0], X_opt[:, 2], '-o', markersize=3, linewidth=2.0, color='#D1495B', label='оптимальная траектория')

for cyl in cfg.cylinders:
    c = plt.Circle((cyl.x, cyl.z), cyl.radius, color='#F4A261', alpha=0.28)
    ax[0].add_patch(c)
for wnd in cfg.windows:
    w = plt.Circle((wnd.x, wnd.z), wnd.radius, color='#2A9D8F', alpha=0.35)
    ax[0].add_patch(w)

ax[0].scatter([cfg.terminal_state[0]], [cfg.terminal_state[2]], marker='o', s=160, color='black', label='целевая точка')
ax[0].set_title('Траектория в плоскости $x_1$–$x_3$')
ax[0].set_xlabel('$x_1$')
ax[0].set_ylabel('$x_3$')
ax[0].axis('equal')
ax[0].legend(loc='best')

ax[1].plot(np.arange(len(trace)) * 100, trace, color='#1D3557')
ax[1].set_title('Ход оптимизации (имитация отжига)')
ax[1].set_xlabel('итерация')
ax[1].set_ylabel('лучший функционал $J$')

plt.tight_layout()
plt.show()

In [ ]:
control_names = ['φ (крен)', 'θ (тангаж)', 'ψ (рыскание)', 'тяга']
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
axes = axes.ravel()

steps = np.arange(1, U_obs.shape[0] + 1)

for j, ax in enumerate(axes):
    ax.step(
        steps,
        best_u_opt[:, j],
        where='post',
        color='#D1495B',
        linewidth=1.3,
        label='оптимизированное',
    )
    ax.plot(steps, best_u_opt[:, j], 'o', color='#D1495B', markersize=3)
    ax.set_title(control_names[j])
    ax.set_xlabel('шаг')
    ax.set_ylabel('значение')

axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
from dynamics import anneal_bundle

eps = 0.1
rng = np.random.default_rng(42)
x0_base = np.zeros(6)
initial_states = x0_base + rng.uniform(-eps, eps, size=(50, 6))  # S=50

best_u, best_j, trace = anneal_bundle(initial_states, best_u_opt, cfg, seed=42)

In [11]:
print(f"J среднее по пучку = {best_j:.4f}")

J среднее по пучку = 9.0182


In [12]:
bundle_states = []
bundle_scores = []

for x0 in initial_states:
    X_k = rollout(x0, best_u, cfg.dt)
    score_k = bolza_cost(x0, best_u, cfg)
    bundle_states.append(X_k)
    bundle_scores.append(score_k)

bundle_states = np.array(bundle_states)   # (S, N+1, 6)
bundle_scores = np.array(bundle_scores)   # (S,)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

for X_k in bundle_states:
    ax.plot(X_k[:, 0], X_k[:, 2], linewidth=1.2, alpha=0.5, color='steelblue')

ax.scatter(initial_states[:, 0], initial_states[:, 2],
           s=25, color='steelblue', zorder=5, label='начальные состояния')

for cyl in cfg.cylinders:
    ax.add_patch(plt.Circle((cyl.x, cyl.z), cyl.radius, color='#F4A261', alpha=0.28))

for wnd in cfg.windows:
    ax.add_patch(plt.Circle((wnd.x, wnd.z), wnd.radius, color='#2A9D8F', alpha=0.45))

ax.scatter([cfg.terminal_state[0]], [cfg.terminal_state[2]],
           marker='o', s=260, color='black', zorder=6, label='целевая точка')

ax.set_title('Траектория пучка')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_3$')
ax.axis('equal')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
control_names  = ['$u_1$ — φ (крен)', '$u_2$ — θ (тангаж)',
                  '$u_3$ — ψ (рыскание)',  '$u_4$ — тяга']
control_limits = [
    (-np.pi/12, np.pi/12),
    (-np.pi,    np.pi),
    (-np.pi/12, np.pi/12),
    (0.0,       12.0),
]

steps = np.arange(1, best_u.shape[0] + 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)

for j, ax in enumerate(axes.ravel()):
    lo, hi = control_limits[j]
    ax.axhline(lo, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.axhline(hi, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.step(steps, best_u[:, j], where='post', color='#D1495B', linewidth=1.5)
    ax.plot(steps, best_u[:, j], 'o', color='#D1495B', markersize=4)
    ax.set_title(control_names[j])
    ax.set_xlabel('шаг')
    ax.set_ylabel('значение')

plt.suptitle('Оптимальное (в среднем) управление $U^*$', fontsize=12)
plt.tight_layout()
plt.show()